# Rollout camera dashboard

One cell (below) renders every camera frame from every rollout run under `rollouts/final/<pool>/episodes/` (пул по умолчанию `pilot_v0`, см. `rollouts/RUNS.md`) — agentview + wrist (where the environment has one). Run it after any `scripts/run_rollouts.py` invocation (smoke-test or full) to eyeball what actually happened without digging through files manually.

Kernel: `slava-notebook` (see README.md). Needs `ipywidgets`/`pillow`/`pandas`, already in `requirements-notebook.txt`.

In [ ]:
import os
import json
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display
from PIL import Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ROLLOUTS_ROOT = PROJECT_ROOT / "rollouts" / "final" / os.environ.get("SLAVA_RUN_POOL", "pilot_v0")
EPISODES_ROOT = ROLLOUTS_ROOT / "episodes"
ANNOTATIONS_PATH = ROLLOUTS_ROOT / "rollout_annotations.jsonl"


def load_annotations():
    if not ANNOTATIONS_PATH.exists():
        return {}
    records = {}
    with open(ANNOTATIONS_PATH, encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            records[record["run_id"]] = record
    return records


def list_runs():
    if not EPISODES_ROOT.exists():
        return []
    return sorted(p.name for p in EPISODES_ROOT.iterdir() if p.is_dir())


def camera_frames(run_id, camera):
    camera_path = EPISODES_ROOT / run_id / "camera" / camera
    if not camera_path.exists():
        return []
    return sorted(camera_path.glob("step_*.png"))


print(f"rollouts root: {ROLLOUTS_ROOT}")
print(f"runs found: {len(list_runs())}")
print(f"annotated episodes: {len(load_annotations())}")

In [ ]:
# --- Dashboard cell: run this after any rollout to browse every recorded camera frame from every run. ---

annotations = load_annotations()
runs = list_runs()

run_dropdown = widgets.Dropdown(options=runs, description="run_id:", layout=widgets.Layout(width="700px"))
step_slider = widgets.IntSlider(value=0, min=0, max=0, step=1, description="step:", continuous_update=False)
info_box = widgets.HTML()
agentview_out = widgets.Output()
wrist_out = widgets.Output()


def render(*_):
    run_id = run_dropdown.value
    if not run_id:
        return
    agent_frames = camera_frames(run_id, "agentview")
    wrist_frames = camera_frames(run_id, "wrist")
    step_slider.max = max(len(agent_frames) - 1, 0)
    idx = min(step_slider.value, step_slider.max)

    record = annotations.get(run_id, {})
    info_box.value = (
        f"<b>{run_id}</b><br>"
        f"model={record.get('model', '?')} variant={record.get('variant', '?')} "
        f"instruction=\"{record.get('instruction', '?')}\"<br>"
        f"success={record.get('success', '?')} failure_type_auto={record.get('failure_type_auto', '?')} "
        f"first_contact_object={record.get('first_contact_object', '?')}<br>"
        f"steps recorded: {len(agent_frames)} (agentview), {len(wrist_frames)} (wrist)"
    )

    agentview_out.clear_output(wait=True)
    with agentview_out:
        if agent_frames:
            display(Image.open(agent_frames[idx]))
        else:
            print("no agentview frames")

    wrist_out.clear_output(wait=True)
    with wrist_out:
        if wrist_frames:
            wrist_idx = min(idx, len(wrist_frames) - 1)
            display(Image.open(wrist_frames[wrist_idx]))
        else:
            print("no wrist camera for this environment/run")


run_dropdown.observe(render, names="value")
step_slider.observe(render, names="value")

display(run_dropdown, step_slider, info_box, widgets.HBox([agentview_out, wrist_out]))
if runs:
    render()

In [ ]:
# Optional: quick contact-sheet grid (every Nth frame) for a single run, for a
# faster skim than stepping through the slider above frame-by-frame.
import math

def contact_sheet(run_id, camera="agentview", every=10, max_frames=24):
    frames = camera_frames(run_id, camera)[::every][:max_frames]
    if not frames:
        print(f"no {camera} frames for {run_id}")
        return
    cols = min(6, len(frames))
    rows = math.ceil(len(frames) / cols)
    thumb = Image.open(frames[0])
    w, h = thumb.size
    sheet = Image.new("RGB", (w * cols, h * rows), "white")
    for i, frame_path in enumerate(frames):
        img = Image.open(frame_path)
        sheet.paste(img, ((i % cols) * w, (i // cols) * h))
    display(sheet)

# Example: contact_sheet(runs[0]) if runs else None